In [0]:
pip install findspark

In [0]:
import findspark, pyspark
from pyspark.sql import SparkSession
findspark.init()
spark = SparkSession.builder.appName('naive_bayes').getOrCreate()

In [0]:
from pyspark.ml.feature import RFormula

In [0]:
abs_path = "/Volumes/workspace/ml_with_spark/raw_data/iris.csv"

iris = spark.read.csv(abs_path, header=True, inferSchema=True)
iris.show(5)

In [0]:
rformula = RFormula(formula='class ~ .', featuresCol='independente', labelCol='dependente')
iris_rf = rformula.fit(iris).transform(iris)
iris_rf.select('independente', 'dependente').show(5, truncate=False)

In [0]:
iris_treino, iris_teste = iris_rf.randomSplit([0.8, 0.2])
print(f'Quantidade de linhas no conjunto de treino: {iris_treino.count()}')
print(f'Quantidade de linhas no conjunto de teste: {iris_teste.count()}')


In [0]:
from pyspark.ml.classification import NaiveBayes

nb = NaiveBayes(
        smoothing=1,
        modelType='multinomial',
        featuresCol='independente',
        labelCol='dependente'
    )

modelo = nb.fit(iris_treino)

In [0]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

previsao = modelo.transform(iris_teste)

avaliador_acc = MulticlassClassificationEvaluator(
    labelCol='dependente',
    predictionCol='prediction',
    metricName='accuracy'
)

acuracia = avaliador_acc.evaluate(previsao)

avaliador_precision = MulticlassClassificationEvaluator(
    labelCol='dependente',
    predictionCol='prediction',
    metricName='weightedPrecision'
)

precision = avaliador_precision.evaluate(previsao)

avaliador_recall = MulticlassClassificationEvaluator(
    labelCol='dependente',
    predictionCol='prediction',
    metricName='weightedRecall'
)

recall = avaliador_recall.evaluate(previsao)



print(f"Acurácia: {acuracia}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")